In [1]:
import json

with open('output/output.json', 'r') as f:
    result = json.load(f)

In [2]:
import matplotlib.pyplot as plt

# 设置中文字体，避免图表中的中文乱码
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False   # 解决负号显示问题

def plot_pie(data, target_name, title, pdf_path, labels=None, **kwargs):
    # 筛选匹配的数据项
    matched = [d for d in data if d.get('name') == target_name]
    if not matched:
        raise ValueError(f"未找到 name 为 '{target_name}' 的数据项")
    if len(matched) > 1:
        print(f"警告: 找到多个 name 为 '{target_name}' 的数据项，将使用第一个。")
    item = matched[0]
    
    # 提取 value 字典，过滤掉值为 0 的类别
    value_dict = item.get('value', {})
    if not value_dict:
        raise ValueError(f"数据项 '{target_name}' 的 value 字段为空")
    
    # 过滤值为 0 的类别（避免饼图中出现占空为 0 的扇区）
    filtered = {k: v for k, v in value_dict.items() if v != 0}
    if not filtered:
        raise ValueError(f"数据项 '{target_name}' 的所有 value 均为 0，无法绘制饼图")
    
    if labels is None:
        labels = list(filtered.keys())
    sizes = list(filtered.values())
    
    # 可选参数设置
    autopct = kwargs.get('autopct', '%1.1f%%')
    startangle = kwargs.get('startangle', 90)
    colors = kwargs.get('colors', None)
    show_legend = kwargs.get('show_legend', True)
    legend_loc = kwargs.get('legend_loc', 'best')
    label_distance = kwargs.get('label_distance', 1.1)
    
    # 创建图形和轴对象
    fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
    
    # 绘制饼图
    wedges, texts, autotexts = ax.pie(
        sizes,
        labels=labels if not show_legend else None,  # 若显示图例，则不直接标在扇区上
        autopct=autopct,
        startangle=startangle,
        colors=colors,
        pctdistance=0.85,          # 百分比文字距离圆心的距离
        labeldistance=label_distance,
        textprops={'fontsize': 10}
    )
    
    # 美化百分比文字
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(9)
        autotext.set_weight('bold')
    
    # 添加图例
    if show_legend:
        # 创建包含 "类别: 占比% " 的图例标签
        total = sum(sizes)
        legend_labels = [f"{l} ({s/total*100:.3f}%)" for l, s in zip(labels, sizes)]
        ax.legend(wedges, legend_labels, title=title, loc=legend_loc, fontsize=9)
    
    # 设置标题
    ax.set_title(title, fontsize=14, pad=20)
    # 确保饼图为正圆
    ax.axis('equal')
    
    # 调整布局并保存为 PDF
    plt.tight_layout()
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.close(fig)
    print(f"饼图已保存至 {pdf_path}")


with open('output/output.json', 'r') as f:
    result = json.load(f)

plot_pie(
    data=result,
    target_name='stats_richi_player_num',
    title='单局立直玩家数分布',
    pdf_path='output/richi_player_num.pdf',
    show_legend=True,
    label_distance=1.15
)

plot_pie(
    data=result,
    target_name='stats_round_end_type',
    title='单局的终局条件',
    pdf_path='output/round_end_type.pdf',
    show_legend=True,
    labels=['流局', '立直荣和', '立直自摸', '门清默听荣和', '门清默听自摸', '副露荣和', '副露自摸'],
    label_distance=1.15
)

饼图已保存至 output/richi_player_num.pdf
饼图已保存至 output/round_end_type.pdf


In [3]:
import json
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False   # 解决负号显示问题

def plot_num_occur_time(data, pdf_path='num_occur_time.pdf', name2title=None,
                        x_label='巡目数', y_label='发生次数', title='n巡目事件发生次数',
                        is_normalize=False):
    # 筛选 type == NUM_OCCUR_TIME
    occur_data = [d for d in data if d['type'] == 'NUM_OCCUR_TIME']
    if name2title is not None:
        occur_data = [d for d in occur_data if d['name'] in name2title]
        present = {d['name'] for d in occur_data}
        missing = set(name2title.keys()) - present
        assert not missing, f"数据中缺少以下名称: {missing}"
    assert len(occur_data) > 0, "没有符合条件的数据"

    # 收集所有键并排序
    all_keys = sorted({int(k) for item in occur_data for k in item['value']})
    x_labels = [str(k) for k in all_keys]

    stats_names, stats_values = [], []
    for item in occur_data:
        legend_name = name2title[item['name']] if name2title else item['name']
        stats_names.append(legend_name)
        stats_values.append([item['value'].get(str(k), 0) for k in all_keys])

    n_groups = len(all_keys)
    n_stats = len(stats_names)
    bar_width = 0.8 / n_stats
    index = np.arange(n_groups)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

    fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
    for i, (vals, name) in enumerate(zip(stats_values, stats_names)):
        weighted_data = [(k, v) for k, v in zip(all_keys, vals) if v > 0]
        total = sum(v for _, v in weighted_data)
        mean_val = sum(k * w for k, w in weighted_data) / total if total >= 1 else 0.0
        std_val = np.sqrt(sum(w * (k - mean_val) ** 2 for k, w in weighted_data) / (total - 1)) if total > 1 else 0.0
        legend_label = f"{name} ({mean_val:.2f}±{std_val:.2f})"

        normalized_vals = vals
        if is_normalize:
            vals_sum = sum(vals)
            normalized_vals = [val / vals_sum for val in vals]

        bars = ax.bar(index + i * bar_width, normalized_vals, bar_width,
                      label=legend_label, color=colors[i % len(colors)])

        for bar, normalized_val, val in zip(bars, normalized_vals, vals):
            if val > 0:
                val_str = '%.3f%% (%d)' % (normalized_val * 100, val) if is_normalize else '%.3f%%' % (normalized_val * 100)
                ax.text(bar.get_x() + bar.get_width() / 2.,
                        bar.get_height() + max(normalized_vals)*0.01,
                        val_str, ha='center', va='bottom',
                        fontsize=6, rotation=90)

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.set_xticks(index + bar_width * (n_stats - 1) / 2)
    ax.set_xticklabels(x_labels)
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.close(fig)
    print(f"图表已保存至 {pdf_path}")


with open('output/output.json', 'r') as f:
    result = json.load(f)


name_map = {
    'stats_game_round': '每个半庄麻将有n小局',
}
plot_num_occur_time(result, 'output/game_round.pdf', name2title=name_map, x_label='局数', y_label='发生比例', title='每个半庄麻将有n小局', is_normalize=True)


name_map = {
    'stats_richi_ok_num': 'n巡目立直成功玩家数',
    'stats_first_richi_ok_num': 'n巡目先制立直成功玩家数',
    'stats_chasing_richi_ok_num': 'n巡目追立直成功玩家数',
    'stats_be_chased_richi_ok_num': 'n巡目立直但被追立直成功玩家数',
}
plot_num_occur_time(result, 'output/richi_num.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生次数', is_normalize=True)


name_map = {
    'stats_agari_richi_ok_dora_num': 'n巡目立直成功宝牌数',
    'stats_agari_first_richi_ok_dora_num': 'n巡目先制立直成功宝牌数',
    'stats_agari_chasing_richi_ok_dora_num': 'n巡目追立直成功宝牌数',
    'stats_agari_be_chased_richi_ok_dora_num': 'n巡目立直但被追立直成功宝牌数',
}
plot_num_occur_time(result, 'output/richi_dora_num.pdf', name2title=name_map, x_label='表宝牌+红宝牌数量', y_label='发生次数', title='n巡目宝牌数-发生次数', is_normalize=True)

图表已保存至 output/game_round.pdf
图表已保存至 output/richi_num.pdf
图表已保存至 output/richi_dora_num.pdf


In [4]:
import json
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False   # 解决负号显示问题

def plot_num_to_mean_std_sample(data, pdf_path='mean_std_sample.pdf', name2title=None,
                                x_label='巡目数', y_label='发生次数', title='n巡目事件发生次数',
                                offset=0.3):
    """
    将 NUM_TO_MEAN_STD_SAMPLE 数据绘制为带误差棒的折线图并保存为 PDF。
    参数 name2title: 字典，键为原始 name，值为图例显示名称。仅绘制 name 在字典中的项。
    参数 offset: 同一列上不同折线的水平偏移总量，默认0.15。设为0则不偏移。
    """
    # 筛选类型
    mean_std_data = [d for d in data if d['type'] == 'NUM_TO_MEAN_STD_SAMPLE']
    if name2title is not None:
        mean_std_data = [d for d in mean_std_data if d['name'] in name2title]
        present = {d['name'] for d in mean_std_data}
        missing = set(name2title.keys()) - present
        assert not missing, f"数据中缺少以下名称: {missing}"
    assert len(mean_std_data) > 0, "没有符合条件的数据"

    # 提取所有巡目并排序
    all_keys_std = sorted({int(k) for item in mean_std_data for k in item['value']})
    x_base = np.array(all_keys_std)          # 原始整数位置
    n_lines = len(mean_std_data)

    # 计算每条线的水平偏移量，使它们均匀分布在基准点左右
    if n_lines > 1 and offset > 0:
        shifts = np.linspace(-offset/2, offset/2, n_lines)
    else:
        shifts = [0.0] * n_lines

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#17becf']
    markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*']

    fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
    for idx, item in enumerate(mean_std_data):
        legend_name = name2title[item['name']] if name2title else item['name']

        # 计算期望begin: deepseek写的，我没改没review
        total_sum = 0.0
        weighted_sum = 0.0
        for k in all_keys_std:
            v = item['value'].get(str(k), None)
            if v is not None and v.get('total', 0) > 0:
                total_sum += v['total']
                weighted_sum += v['mean'] * v['total']
        if total_sum > 0:
            expected = weighted_sum / total_sum
            legend_name = f"{legend_name} ({expected:.2f})"
        # 计算期望end
        
        x_pos = x_base + shifts[idx]          # 偏移后的横坐标
        means, yerrs = [], []
        for k in all_keys_std:
            v = item['value'].get(str(k), None)
            if v is not None:
                means.append(v['mean'])
                sem = v['std'] / np.sqrt(v['total']) if v['total'] > 0 else 0
                yerrs.append(sem)
            else:
                means.append(np.nan)
                yerrs.append(0)

        ax.errorbar(x_pos, means, yerr=yerrs,
                    fmt=markers[idx % len(markers)] + '-',
                    color=colors[idx % len(colors)],
                    label=legend_name,
                    capsize=3, linewidth=1.5, markersize=5)

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.set_xticks(x_base)                     # 刻度仍在原始整数位置
    ax.set_xticklabels(all_keys_std)
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.close(fig)
    print(f"图表已保存至 {pdf_path}")


with open('output/output.json', 'r') as f:
    result = json.load(f)


name_map = {
    'stats_richi_n_ron_rate': 'n巡立直荣和率',
    'stats_richi_n_tsumo_rate': 'n巡立直自摸率',
    'stats_richi_n_be_ron_rate': 'n巡立直被荣和率',
    'stats_richi_n_be_tsumo_rate': 'n巡立直被自摸率',
    'stats_richi_n_draw_rate': 'n巡立直横移动率',
    'stats_richi_n_ryuukyoku_rate': 'n巡立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_oya_richi_n_ron_rate': 'n巡庄家立直荣和率',
    'stats_oya_richi_n_tsumo_rate': 'n巡庄家立直自摸率',
    'stats_oya_richi_n_be_ron_rate': 'n巡庄家立直被荣和率',
    'stats_oya_richi_n_be_tsumo_rate': 'n巡庄家立直被自摸率',
    'stats_oya_richi_n_draw_rate': 'n巡庄家立直横移动率',
    'stats_oya_richi_n_ryuukyoku_rate': 'n巡庄家立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/oya_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_not_oya_richi_n_ron_rate': 'n巡闲家立直荣和率',
    'stats_not_oya_richi_n_tsumo_rate': 'n巡闲家立直自摸率',
    'stats_not_oya_richi_n_be_ron_rate': 'n巡闲家立直被荣和率',
    'stats_not_oya_richi_n_be_tsumo_rate': 'n巡闲家立直被自摸率',
    'stats_not_oya_richi_n_draw_rate': 'n巡闲家立直横移动率',
    'stats_not_oya_richi_n_ryuukyoku_rate': 'n巡闲家立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/not_oya_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_richi_n_gain': 'n巡立直局收支',
    'stats_richi_n_ron_gain': 'n巡立直荣和局收支',
    'stats_richi_n_tsumo_gain': 'n巡立直自摸局收支',
    'stats_richi_n_be_ron_gain': 'n巡立直被荣和局收支',
    'stats_richi_n_be_tsumo_gain': 'n巡立直被自摸局收支',
    'stats_richi_n_draw_gain': 'n巡立直横移动局收支',
    'stats_richi_n_ryuukyoku_gain': 'n巡立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_oya_richi_n_gain': 'n巡庄家立直局收支',
    'stats_oya_richi_n_ron_gain': 'n巡庄家立直荣和局收支',
    'stats_oya_richi_n_tsumo_gain': 'n巡庄家立直自摸局收支',
    'stats_oya_richi_n_be_ron_gain': 'n巡庄家立直被荣和局收支',
    'stats_oya_richi_n_be_tsumo_gain': 'n巡庄家立直被自摸局收支',
    'stats_oya_richi_n_draw_gain': 'n巡庄家立直横移动局收支',
    'stats_oya_richi_n_ryuukyoku_gain': 'n巡庄家立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/oya_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_not_oya_richi_n_gain': 'n巡闲家立直局收支',
    'stats_not_oya_richi_n_ron_gain': 'n巡闲家立直荣和局收支',
    'stats_not_oya_richi_n_tsumo_gain': 'n巡闲家立直自摸局收支',
    'stats_not_oya_richi_n_be_ron_gain': 'n巡闲家立直被荣和局收支',
    'stats_not_oya_richi_n_be_tsumo_gain': 'n巡闲家立直被自摸局收支',
    'stats_not_oya_richi_n_draw_gain': 'n巡闲家立直横移动局收支',
    'stats_not_oya_richi_n_ryuukyoku_gain': 'n巡闲家立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/not_oya_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_first_richi_n_ron_rate': 'n巡先制立直荣和率',
    'stats_first_richi_n_tsumo_rate': 'n巡先制立直自摸率',
    'stats_first_richi_n_be_ron_rate': 'n巡先制立直被荣和率',
    'stats_first_richi_n_be_tsumo_rate': 'n巡先制立直被自摸率',
    'stats_first_richi_n_draw_rate': 'n巡先制立直横移动率',
    'stats_first_richi_n_ryuukyoku_rate': 'n巡先制立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/first_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_first_richi_n_gain': 'n巡先制立直局收支',
    'stats_first_richi_n_ron_gain': 'n巡先制立直荣和局收支',
    'stats_first_richi_n_tsumo_gain': 'n巡先制立直自摸局收支',
    'stats_first_richi_n_be_ron_gain': 'n巡先制立直被荣和局收支',
    'stats_first_richi_n_be_tsumo_gain': 'n巡先制立直被自摸局收支',
    'stats_first_richi_n_draw_gain': 'n巡先制立直横移动局收支',
    'stats_first_richi_n_ryuukyoku_gain': 'n巡先制立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/first_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_chasing_richi_n_ron_rate': 'n巡追立直荣和率',
    'stats_chasing_richi_n_tsumo_rate': 'n巡追立直自摸率',
    'stats_chasing_richi_n_be_ron_rate': 'n巡追立直被荣和率',
    'stats_chasing_richi_n_be_tsumo_rate': 'n巡追立直被自摸率',
    'stats_chasing_richi_n_draw_rate': 'n巡追立直横移动率',
    'stats_chasing_richi_n_ryuukyoku_rate': 'n巡追立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/chasing_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_chasing_richi_n_gain': 'n巡追立直局收支',
    'stats_chasing_richi_n_ron_gain': 'n巡追立直荣和局收支',
    'stats_chasing_richi_n_tsumo_gain': 'n巡追立直自摸局收支',
    'stats_chasing_richi_n_be_ron_gain': 'n巡追立直被荣和局收支',
    'stats_chasing_richi_n_be_tsumo_gain': 'n巡追立直被自摸局收支',
    'stats_chasing_richi_n_draw_gain': 'n巡追立直横移动局收支',
    'stats_chasing_richi_n_ryuukyoku_gain': 'n巡追立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/chasing_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')


name_map = {
    'stats_be_chased_richi_n_ron_rate': 'n巡被追立直荣和率',
    'stats_be_chased_richi_n_tsumo_rate': 'n巡被追立直自摸率',
    'stats_be_chased_richi_n_be_ron_rate': 'n巡被追立直被荣和率',
    'stats_be_chased_richi_n_be_tsumo_rate': 'n巡被追立直被自摸率',
    'stats_be_chased_richi_n_draw_rate': 'n巡被追立直横移动率',
    'stats_be_chased_richi_n_ryuukyoku_rate': 'n巡被追立直流局率',
}
plot_num_to_mean_std_sample(result, 'output/be_chased_richi_result_rate.pdf', name2title=name_map, x_label='巡目数', y_label='发生次数', title='n巡目事件发生概率（带误差棒）')


name_map = {
    'stats_be_chased_richi_n_gain': 'n巡被追立直局收支',
    'stats_be_chased_richi_n_ron_gain': 'n巡被追立直荣和局收支',
    'stats_be_chased_richi_n_tsumo_gain': 'n巡被追立直自摸局收支',
    'stats_be_chased_richi_n_be_ron_gain': 'n巡被追立直被荣和局收支',
    'stats_be_chased_richi_n_be_tsumo_gain': 'n巡被追立直被自摸局收支',
    'stats_be_chased_richi_n_draw_gain': 'n巡被追立直横移动局收支',
    'stats_be_chased_richi_n_ryuukyoku_gain': 'n巡被追立直流局局收支',
}
plot_num_to_mean_std_sample(result, 'output/be_chased_richi_result_gain.pdf', name2title=name_map, x_label='巡目数', y_label='点棒差', title='n巡目事件发生时点棒差（带误差棒）')

图表已保存至 output/richi_result_rate.pdf
图表已保存至 output/oya_richi_result_rate.pdf
图表已保存至 output/not_oya_richi_result_rate.pdf
图表已保存至 output/richi_result_gain.pdf
图表已保存至 output/oya_richi_result_gain.pdf
图表已保存至 output/not_oya_richi_result_gain.pdf
图表已保存至 output/first_richi_result_rate.pdf
图表已保存至 output/first_richi_result_gain.pdf
图表已保存至 output/chasing_richi_result_rate.pdf
图表已保存至 output/chasing_richi_result_gain.pdf
图表已保存至 output/be_chased_richi_result_rate.pdf
图表已保存至 output/be_chased_richi_result_gain.pdf


In [7]:
import json
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False

ALL_TILES = [
    '1m','2m','3m','4m','5m','6m','7m','8m','9m',
    '1p','2p','3p','4p','5p','6p','7p','8p','9p',
    '1s','2s','3s','4s','5s','6s','7s','8s','9s',
    '1z','2z','3z','4z','5z','6z','7z',
]


def plot_bit_set_occur_time(data, pdf_path, name2title=None,
                            from_round=None, to_round=None,
                            x_label='牌', y_label='铳率',
                            title='立直听牌铳率'):
    """
    将 BIT_SET_TO_OCCUR_TIME 数据绘制为直方图。
    外层 key 为巡目数，内层 key 为组合听牌（如 "3m6m"），value 为出现次数。
    按 from_round/to_round 筛选巡目范围后，合并内层数据，统计每张牌的铳率。
    """
    bit_data = [d for d in data if d['type'] == 'BIT_SET_TO_OCCUR_TIME']
    if name2title is not None:
        bit_data = [d for d in bit_data if d['name'] in name2title]
        present = {d['name'] for d in bit_data}
        missing = set(name2title.keys()) - present
        assert not missing, f"数据中缺少以下名称: {missing}"
    assert len(bit_data) > 0, "没有符合条件的数据"

    n_tiles = len(ALL_TILES)
    stats_names, stats_values = [], []
    for item in bit_data:
        legend_name = name2title[item['name']] if name2title else item['name']

        # 合并符合巡目范围的各层数据
        aggregated = {}
        for round_str, inner_dict in item['value'].items():
            r = int(round_str)
            if (from_round is None or r >= from_round) and \
               (to_round is None or r <= to_round):
                for compound_key, count in inner_dict.items():
                    aggregated[compound_key] = aggregated.get(compound_key, 0) + count

        total_riichi = sum(aggregated.values())

        # 每张牌的铳率
        tile_counts = np.zeros(n_tiles, dtype=np.float64)
        for tile_idx, tile_name in enumerate(ALL_TILES):
            for compound_key, count in aggregated.items():
                if tile_name in compound_key:
                    tile_counts[tile_idx] += count
        tile_rates = tile_counts / total_riichi if total_riichi > 0 else tile_counts

        legend_label = f"{legend_name} (总场次: {total_riichi})"
        stats_names.append(legend_label)
        stats_values.append((tile_rates, tile_counts, total_riichi))

    # 绘图
    n_stats = len(stats_names)
    bar_width = 0.8 / n_stats
    index = np.arange(n_tiles)
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

    fig, ax = plt.subplots(figsize=(18, 6), dpi=150)
    for i, (name, (rates, counts, total)) in enumerate(zip(stats_names, stats_values)):
        bars = ax.bar(index + i * bar_width, rates, bar_width,
                      label=name, color=colors[i % len(colors)])

        max_rate = max(rates) if max(rates) > 0 else 1
        for bar, rate, cnt in zip(bars, rates, counts):
            if cnt > 0:
                ax.text(bar.get_x() + bar.get_width() / 2.,
                        bar.get_height() + max_rate * 0.01,
                        '%.2f%%(%d)' % (rate * 100, int(cnt)),
                        ha='center', va='bottom',
                        fontsize=5.5, rotation=90)

    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.set_xticks(index + bar_width * (n_stats - 1) / 2)
    ax.set_xticklabels(ALL_TILES, fontsize=8)
    ax.legend(fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight', format='pdf')
    plt.close(fig)
    print(f"图表已保存至 {pdf_path}")


with open('output/output.json', 'r') as f:
    result = json.load(f)

round_ranges = [
    (None, None, 'richi_tenpai_rate',                    '全部巡目'),
    (1,    1,    'richi_tenpai_rate_from_1_to_1',        '1巡'),
    (2,    3,    'richi_tenpai_rate_from_2_to_3',        '2-3巡'),
    (4,    6,    'richi_tenpai_rate_from_4_to_6',        '4-6巡'),
    (7,    9,    'richi_tenpai_rate_from_7_to_9',        '7-9巡'),
    (10,   12,   'richi_tenpai_rate_from_10_to_12',      '10-12巡'),
    (13,   15,   'richi_tenpai_rate_from_13_to_15',      '13-15巡'),
    (16,   None, 'richi_tenpai_rate_from_16_to_inf',     '≥16巡'),
]

for from_r, to_r, pdf_name, range_label in round_ranges:
    plot_bit_set_occur_time(
        result,
        f'output/{pdf_name}.pdf',
        name2title={'stats_richi_tenpai_content': f'立直听牌铳率 ({range_label})'},
        from_round=from_r,
        to_round=to_r,
        x_label='牌',
        y_label='铳率',
        title=f'立直听牌铳率 ({range_label})',
    )


图表已保存至 output/richi_tenpai_rate.pdf
图表已保存至 output/richi_tenpai_rate_from_1_to_1.pdf
图表已保存至 output/richi_tenpai_rate_from_2_to_3.pdf
图表已保存至 output/richi_tenpai_rate_from_4_to_6.pdf
图表已保存至 output/richi_tenpai_rate_from_7_to_9.pdf
图表已保存至 output/richi_tenpai_rate_from_10_to_12.pdf
图表已保存至 output/richi_tenpai_rate_from_13_to_15.pdf
图表已保存至 output/richi_tenpai_rate_from_16_to_inf.pdf


In [ ]:
import json
import re
from collections import defaultdict

# ─── 解析与规范化 ─────────────────────────────────────────

TILE_RE = re.compile(r'\d+[mpsz]')

# 字牌分组
Z_A = {'1z','2z','3z','4z'}
Z_B = {'5z','6z','7z'}


def parse_key(compound_key):
    """ "3m6m" → [('m',3), ('m',6)] 排序返回 """
    tiles = [(t[-1], int(t[:-1])) for t in TILE_RE.findall(compound_key)]
    tiles.sort()
    return tiles


def mirror_fold(nums):
    """ 序数序列，比镜像字典序，取更小的 """
    nums_sorted = sorted(nums)
    mirrored = sorted(10 - n for n in nums_sorted)
    if mirrored < nums_sorted:
        return mirrored
    return nums_sorted


def compress_z(z_tiles):
    """ 字牌压缩：A组连续1z开始，B组连续5z开始 """
    a_count = sum(1 for t in z_tiles if t in Z_A)
    b_count = sum(1 for t in z_tiles if t in Z_B)
    result = []
    for i in range(a_count):
        result.append(f'{i+1}z')
    for i in range(b_count):
        result.append(f'{i+5}z')
    return result


def normalize(compound_key):
    """ 规范化一个听牌 key，返回标准化后的字符串 """
    tiles = parse_key(compound_key)  # [(suit, num), ...], sorted

    # 分组
    suit_tiles = defaultdict(list)  # 'm'/'p'/'s' -> [num, ...]
    z_tiles = []
    for suit, num in tiles:
        if suit == 'z':
            z_tiles.append(f'{num}z')
        else:
            suit_tiles[suit].append(num)

    # Step 2: 每种序数花色做镜像折叠
    normalized_suits = {}
    for suit, nums in suit_tiles.items():
        normalized_suits[suit] = mirror_fold(nums)

    # Step 3: 字牌压缩
    z_result = compress_z(z_tiles)

    # Step 4: 花色归一化 — 按序列字典序排序
    suit_names = {'m', 'p', 's'}
    ordered = sorted(normalized_suits.items(), key=lambda kv: kv[1])  # by seq

    result_parts = []
    for i, (_, nums) in enumerate(ordered):
        # 按顺序分配 m / p / s
        new_suit = ['m','p','s'][i]
        for num in sorted(nums):
            result_parts.append(f'{num}{new_suit}')

    # 字牌追加在后面
    result_parts.extend(z_result)

    return ''.join(result_parts)


# ─── 分析 ─────────────────────────────────────────────────

def analyze_tenpai_patterns(data, from_round=None, to_round=None, top_n=30):
    """
    从 BIT_SET_TO_OCCUR_TIME 数据中，按巡目范围筛选，规范化听牌 key，
    合并频次，返回 (total, [(normalized_key, count, pct), ...]) 前 top_n。
    """
    # 找到 BIT_SET_TO_OCCUR_TIME 类型的数据
    bit_data = [d for d in data if d['type'] == 'BIT_SET_TO_OCCUR_TIME']
    assert bit_data, "没有 BIT_SET_TO_OCCUR_TIME 数据"

    # 合并符合巡目范围的各层
    aggregated = defaultdict(int)
    for round_str, inner_dict in bit_data[0]['value'].items():
        r = int(round_str)
        if (from_round is None or r >= from_round) and \
           (to_round is None or r <= to_round):
            for compound_key, count in inner_dict.items():
                aggregated[compound_key] += count

    # 规范化 → 合并
    normalized_counts = defaultdict(int)
    for compound_key, count in aggregated.items():
        norm_key = normalize(compound_key)
        normalized_counts[norm_key] += count

    total = sum(normalized_counts.values())

    # 排序，取前 N
    sorted_items = sorted(normalized_counts.items(), key=lambda kv: -kv[1])
    top = [(key, cnt, cnt/total*100) for key, cnt in sorted_items[:top_n]]

    return total, top


# ─── 输出 ─────────────────────────────────────────────────

with open('output/output.json', 'r') as f:
    result = json.load(f)

round_ranges = [
    (None, None, '全部巡目'),
    (1,    1,    '1巡'),
    (2,    3,    '2-3巡'),
    (4,    6,    '4-6巡'),
    (7,    9,    '7-9巡'),
    (10,   12,   '10-12巡'),
    (13,   15,   '13-15巡'),
    (16,   None, '≥16巡'),
]

for from_r, to_r, label in round_ranges:
    total, top = analyze_tenpai_patterns(result, from_round=from_r, to_round=to_r, top_n=-1)
    print(f'\n=== {label} === 总立直数: {total}')
    # result2 = {
    #     '两面好型': 0,
    #     '三面好型': 0,
    #     '单骑幺九': 0,
    #     '序数双碰': 0,
    #     '序数和字双碰': 0,
    #     '字双碰': 0,
    #     '顺子愚型': 0,
    #     '其他': 0,
    # }
    # for rank, (key, cnt, pct) in enumerate(top, 1):
    #     set_两面好型 = set({'1m4m', '2m5m', '3m6m'})
    #     set_三面好型 = set({'1m4m7m', '2m5m8m'})
    #     set_单骑幺九 = set({'1m', '1z', '5z'})
    #     set_序数双碰 = set({f'{num1}m{num2}m' for num1 in range(1, 10) for num2 in range(1, 10) if num1 < num2 and num2 != num1 + 3}) | set({f'{num1}m{num2}p' for num1 in range(1, 10) for num2 in range(1, 10)})
    #     set_序数和字双碰 = set({f'{num1}m{num2}z' for num1 in range(1, 10) for num2 in range(1, 8)})
    #     set_字双碰 = set({f'{num1}z{num2}z' for num1 in range(1, 8) for num2 in range(1, 8) if num1 < num2})
    #     set_顺子愚型 = set({'2m', '3m', '4m', '5m'})
    #     set_顺子愚型 = set({'2m', '3m', '4m', '5m'})
    #     if key in set_两面好型:
    #         result2['两面好型'] += pct
    #     elif key in set_三面好型:
    #         result2['三面好型'] += pct
    #     elif key in set_单骑幺九:
    #         result2['单骑幺九'] += pct
    #     elif key in set_序数双碰:
    #         result2['序数双碰'] += pct
    #     elif key in set_序数和字双碰:
    #         result2['序数和字双碰'] += pct
    #     elif key in set_字双碰:
    #         result2['字双碰'] += pct
    #     elif key in set_顺子愚型:
    #         result2['顺子愚型'] += pct
    #     else:
    #         result2['其他'] += pct
    # for k, v in result2.items():
    #     print(f'{k:20s} {v:.2f}%')

    for rank, (key, cnt, pct) in enumerate(top, 1):
        print(f'{rank:2d}. {key:30s} {cnt:6d} ({pct:.2f}%)')



=== 全部巡目 === 总立直数: 1340085
两面好型                 54.24%
三面好型                 5.67%
单骑幺九                 3.43%
序数双碰                 6.12%
序数和字双碰               3.94%
字双碰                  0.38%
顺子愚型                 23.51%
其他                   2.72%

=== 1巡 === 总立直数: 4680
两面好型                 29.27%
三面好型                 2.86%
单骑幺九                 15.09%
序数双碰                 6.26%
序数和字双碰               6.15%
字双碰                  1.09%
顺子愚型                 38.29%
其他                   0.96%

=== 2-3巡 === 总立直数: 37695
两面好型                 45.80%
三面好型                 4.66%
单骑幺九                 7.29%
序数双碰                 5.76%
序数和字双碰               6.88%
字双碰                  1.05%
顺子愚型                 26.54%
其他                   2.01%

=== 4-6巡 === 总立直数: 274290
两面好型                 51.07%
三面好型                 5.40%
单骑幺九                 4.00%
序数双碰                 5.71%
序数和字双碰               5.14%
字双碰                  0.55%
顺子愚型                 25.71%
其他                   2.41%

=== 7-9巡 === 总立直数: 488